In [ ]:
import os 
import sys
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
from scipy.sparse import csr_matrix
import pandas as pd
import anndata as ad
#from anndata.experimental import concat_on_disk


In [ ]:
import yaml

base_path = Path('../..').resolve()
sys.path.append(str(base_path))
from helpers import singlecell_utils

with open(base_path / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

In [ ]:
GENE_H5AD = base_path / cfg['gene_h5ad']
GENE_H5AD

In [ ]:
COUNTS_DIR = '../counts/'
MERGED_COUNTS_DIR = Path('../singlecell_counts/')
MERGED_H5AD_TEMP_PATH = MERGED_COUNTS_DIR / Path(GENE_H5AD).name.replace('.h5ad', '_PAS_NoPASinfo.h5ad')
MERGED_H5AD_PATH = MERGED_COUNTS_DIR / Path(GENE_H5AD).name.replace('.h5ad', '_PAS.h5ad')

NCPU = 8

MERGED_COUNTS_DIR.mkdir(exist_ok=True)

# Read bed metadata for var table

In [ ]:
# They are all the same (see 0_check_scapturequant_bedfiles.sh), so read only one of them
bedmeta_path = sorted(Path(COUNTS_DIR).glob('*/*.KeepPAS.metadata'))[0]
df_bedmeta = pd.read_table(bedmeta_path, index_col=None, header=None, dtype={0:'category'})

print(len(df_bedmeta))
df_bedmeta.head()

In [ ]:
# Differences between gene names and PAS names: only should be the PAS numbers
print(set(df_bedmeta[[3, 12]].apply(lambda x: x[3].replace(x[12], ''), axis=1)))

In [ ]:
pas_to_gene = dict(zip(df_bedmeta[3], df_bedmeta[12]))

In [ ]:
df_var = df_bedmeta[[3, 12]].copy()
df_var.columns = ('PAS_id', 'gene_name')
df_var.set_index('PAS_id', inplace=True)
df_var

# Read the entire barcode table to get the list of files and cells

In [ ]:
ad_gene = singlecell_utils.read_everything_but_X(GENE_H5AD)

In [ ]:
df_obs = ad_gene.obs
df_obs.head()

In [ ]:
df_obs['Channel']

In [ ]:
# Before this, free some memory
del ad_gene

In [ ]:
df_obs['cell_barcode'] = df_obs.index.str.split('_').map(lambda x: x[-1])

## Check : is barcode prefix always the same as channel?

In [ ]:
df_obs['cell_barcode_prefix'] = df_obs.index.str.split('_').map(lambda x: '_'.join(x[:-1]))

print(all(df_obs['cell_barcode_prefix'] == df_obs['Channel']))
del df_obs['cell_barcode_prefix']

# Test processing a single file

In [ ]:
channel = df_obs['Channel'].iloc[0]

df_cnt = pd.read_table(f'{COUNTS_DIR}{channel}/{channel}.KeepCell.UMIs.tsv.gz', index_col=0)
print(len(df_cnt))
df_cnt.iloc[:5,:5]

In [ ]:
barcode_key_cnt = channel + '_' + df_cnt.columns.str.split('-').str[0] # No -1, -2 in GENESIS
df_cnt.columns = barcode_key_cnt
df_cnt.iloc[:5,:5]

In [ ]:
df_obs_cut = df_obs.loc[barcode_key_cnt]
len(df_obs_cut)

In [ ]:
adata = ad.AnnData(csr_matrix(df_cnt.T.values), obs=df_obs_cut, var=df_var)

In [ ]:
adata.write_h5ad('test.h5ad', compression='gzip')

## Test reading test h5ad

In [ ]:
dat_test = ad.read_h5ad('test.h5ad')
dat_test

In [ ]:
dat_test.obs.head()

In [ ]:
dat_test.var.head()

In [ ]:
dat_test.X[:10, :10].todense().T

In [ ]:
(df_cnt == dat_test.X.todense().T).all().all()

# Read all PAS count

In [ ]:
df_samples = df_obs[['Channel']].drop_duplicates()
print(len(df_samples))
# Remove unprocessed
df_samples = df_samples[df_samples.apply(lambda x: os.path.exists(f'{COUNTS_DIR}{x.Channel}/{x.Channel}.KeepCell.UMIs.tsv.gz'), axis=1)].copy()
print(len(df_samples))

In [ ]:
def process_single_tsv_to_h5ad(channel):
    tsv_path = f'{COUNTS_DIR}{channel}/{channel}.KeepCell.UMIs.tsv.gz'
    h5ad_path = f'{COUNTS_DIR}{channel}/{channel}.h5ad'
    
    df_cnt = pd.read_table(tsv_path, index_col=0)

    barcode_key_cnt = channel + '_' + df_cnt.columns.str.split('-').str[0] # No -1, -2 in GENESIS
    df_obs_cut = df_obs.loc[barcode_key_cnt]
    df_cnt.columns = barcode_key_cnt
   
    adata = ad.AnnData(csr_matrix(df_cnt.T.values), obs=df_obs_cut, var=df_var)
    adata.write_h5ad(h5ad_path, compression='gzip')
    return h5ad_path

with tqdm(total=len(df_samples)) as pbar:
    with ProcessPoolExecutor(max_workers=NCPU) as executor:
        futures = []
        for channel in df_samples['Channel']:
            fut = executor.submit(process_single_tsv_to_h5ad, channel)
            futures.append(fut)
            
        for fut in as_completed(futures):
            pbar.update(n=1)  # Increments counter
            print(f'\r{fut.result()} is processed!                ', end='', file=sys.stdout)
    


# Ondisk merge of h5ad files

In [ ]:
each_h5ad_paths = [f'{COUNTS_DIR}{channel}/{channel}.h5ad' for channel in df_samples['Channel']]
len(each_h5ad_paths)

In [ ]:
with tqdm(total=len(df_samples)) as pbar:
    singlecell_utils.concat_on_disk(each_h5ad_paths, MERGED_H5AD_TEMP_PATH, pbar=pbar, read_verbose=False, x_dtype=np.int64)

# Read merged h5ad file

In [ ]:
dat_merged = ad.read_h5ad(MERGED_H5AD_TEMP_PATH)
dat_merged.obs.head()

In [ ]:
dat_merged.X

# Write pseudobulks as csv format, to be used in R

In [ ]:
df_obs.head().T

In [ ]:
cols = ['class', 'subclass', 'subtype']
df_obs[cols].drop_duplicates().sort_values(by=cols)

In [ ]:
chk = df_obs.groupby('subtype')[['class', 'subclass']].nunique()
chk[(chk > 1).any(axis=1)]

In [ ]:
PSB_DIR = Path('../psb/')
PSB_CNT_DIR = PSB_DIR / f'count_no_cutoff'

CLASS_DIR = PSB_CNT_DIR / 'class'
SUBCLASS_DIR = PSB_CNT_DIR / 'subclass'
SUBTYPE_DIR = PSB_CNT_DIR / 'subtype'

CLASS_DIR.mkdir(exist_ok=True, parents=True)
SUBCLASS_DIR.mkdir(exist_ok=True, parents=True)
SUBTYPE_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
# ignore performance warning
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

def write_single_csv(cell_type, level, out_dir):
    out_path = out_dir / f'PAS_read_count_{cell_type}_Channel.csv.gz'
    
    dat_cut = dat_merged[dat_merged.obs[level]==cell_type]
    #dat_cut = dat_cut[:, se_pas_to_keep]
    df_psb = singlecell_utils.get_pseudobulk(dat_cut, colname='Channel')
    
    missed_cols = [chn for chn in dat_merged.obs['Channel'].unique() if chn not in df_psb]
    df_psb = df_psb.assign(**dict.fromkeys(missed_cols, 0))
    df_psb[sorted(df_psb.columns)].to_csv(out_path, compression='gzip')
    
    return f'{cell_type} {(dat_merged.obs[level]==cell_type).sum()}'

def run_write_csv(level, out_dir):
    with tqdm(total=len(dat_merged.obs[level].unique())) as pbar:
        with ProcessPoolExecutor(max_workers=NCPU) as executor:
            futures = []
            for cell_type in dat_merged.obs[level].unique():
                fut = executor.submit(write_single_csv, cell_type, level, out_dir)
                futures.append(fut)
    
            for fut in as_completed(futures):
                pbar.update(n=1)  # Increments counter
                print(fut.result())
            
run_write_csv('class', CLASS_DIR)

In [ ]:
run_write_csv('subclass', SUBCLASS_DIR)

In [ ]:
run_write_csv('subtype', SUBTYPE_DIR)

# Write var (PAS) info to separate table, and add var to merged h5ad also

In [ ]:
dat_merged.var.head()

In [ ]:
df_var.head()

In [ ]:
df_bedmeta.head()

## Fix bedmeta column names

In [ ]:
df_bedmeta.columns = 'chr start end pas_name score strand thick_start thick_end color block_count block_sizes block_starts gene_name pas_group'.split()
df_bedmeta.head()

## Write into files

In [ ]:
df_bedmeta.index = df_bedmeta['pas_name']
var_columns = 'chr start end gene_name strand block_count block_sizes block_starts pas_group'.split()
df_bedmeta[var_columns].head()

In [ ]:
for col in var_columns:
    df_var[col] = df_bedmeta[col]

### Write entire PAS info, to h5ad and csv

In [ ]:
for col in df_var:
    dat_merged.var[col] = df_var[col]

dat_merged.var

In [ ]:
# Write h5ad
dat_merged.write_h5ad(MERGED_COUNTS_DIR / MERGED_H5AD_PATH, compression='gzip')

# Write PAS table:
df_var.to_csv(MERGED_COUNTS_DIR / f'PAS_info_for_rowdata_all.csv.gz', compression='gzip')

In [ ]:
# Remove h5ad uncompressed, without PAS info
MERGED_H5AD_TEMP_PATH.unlink()